# K Sweep — Plan B (HMM multivariate M)

Entrena caches HMM-M compartidas entre canales y ejecuta el *K sweep* RITMO-M. Todo es resumible: salta caches/resultados ya existentes.

**Este notebook envuelve dos scripts standalone** (la fuente canónica de verdad):
- `scripts/plan_b/train_hmm_caches.py` — entrena caches `cache/hmm_M_{dataset}_K{k}.pth`.
- `scripts/plan_b/run_ritmo_sweep.py` — invoca `run.py --task_name plan_b` para cada (K, variante, horizonte, dataset).

Si prefieres correrlo sin notebook (recomendado en WSL CPU por tiempos largos):
```bash
conda activate ritmo
cd /home/jaime/TFG/RITMO
nohup python -u scripts/plan_b/train_hmm_caches.py > logs/plan_b_caches.log 2>&1 &
# Cuando termine (o interrumpas y vuelvas):
nohup python -u scripts/plan_b/run_ritmo_sweep.py > logs/plan_b_sweep.log 2>&1 &
tail -f logs/plan_b_sweep.log
```

In [ ]:
# Celda 1. Imports + workdir + config desde scripts/plan_b/plan_b_config.py
import os, sys, subprocess
from pathlib import Path
REPO = '/home/jaime/TFG/RITMO'
os.chdir(REPO)
if REPO not in sys.path: sys.path.insert(0, REPO)

from scripts.plan_b.plan_b_config import (
    DATASETS, K_VALUES_BY_DATASET, VARIANTS, HORIZONS, SEED,
    cache_path, experiment_tag,
)
Path('logs').mkdir(exist_ok=True)
print('Datasets:', [d['name'] for d in DATASETS])
print('Ks:', K_VALUES_BY_DATASET)
print('Variants:', VARIANTS, 'Horizons:', HORIZONS)

In [ ]:
# Celda 2. Entrenar caches HMM-M (resumible). Lanza el script standalone.
# Aprox. tiempos CPU: ETTh1/h2 ~1-3 min por K, Weather ~3-8 min, Electricity ~15-40 min.
cmd = ['python', '-u', 'scripts/plan_b/train_hmm_caches.py']
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=False)

In [ ]:
# Celda 3. K sweep RITMO-M (resumible). Lanza el script standalone.
# Tip: puedes acotar con --only-dataset / --horizons / --ks si quieres partir la ejecucion.
cmd = ['python', '-u', 'scripts/plan_b/run_ritmo_sweep.py']
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=False)

In [ ]:
# Celda 4. Tabla resumen: MSE/MAE por (dataset, variante, K, pred_len) + K optimo.
import numpy as np, pandas as pd, re
RESULTS = Path('results')
rows = []
for p in sorted(RESULTS.glob('plan_b_PLANB_*_0')):
    m = (p / 'metrics.npy')
    if not m.exists(): continue
    mae, mse, rmse, mape, mspe = np.load(m)
    # setting = plan_b_{tag}_TransformerCommon_..._{tag}_0 donde tag = PLANB_{ds}_{variant}_K{K}_pl{pl}
    m_tag = re.search(r'PLANB_([^_]+)_(hmm_soft_residual|hmm_soft|hmm_augmented|hmm_split|hmm_patched|hmm)_K(\d+)_pl(\d+)', p.name)
    if not m_tag:
        print(f'skip (tag): {p.name}'); continue
    ds, variant, K, pl = m_tag.group(1), m_tag.group(2), int(m_tag.group(3)), int(m_tag.group(4))
    rows.append({'dataset': ds, 'variant': variant, 'K': K, 'pred_len': pl,
                 'MSE': float(mse), 'MAE': float(mae)})

df = pd.DataFrame(rows)
print(f'Experimentos cargados: {len(df)}')
if len(df) == 0:
    raise SystemExit('Sin resultados todavia. Ejecutar Celda 3 o lanzar run_ritmo_sweep.py.')

df = df.sort_values(['dataset', 'variant', 'K', 'pred_len']).reset_index(drop=True)
print('\n-- Resultados ordenados --')
print(df.to_string(index=False))

avg = df.groupby(['dataset', 'variant', 'K'])['MSE'].mean().reset_index(name='MSE_avg')
best = avg.loc[avg.groupby('dataset')['MSE_avg'].idxmin()].reset_index(drop=True)
print('\n-- Configuracion optima Plan B (min MSE promedio sobre horizontes disponibles) --')
print(best.to_string(index=False))

df.to_csv(RESULTS / 'plan_b_k_sweep.csv', index=False)
best.to_csv(RESULTS / 'plan_b_best_config.csv', index=False)
print(f"\nGuardado: {RESULTS/'plan_b_k_sweep.csv'}")
print(f"Guardado: {RESULTS/'plan_b_best_config.csv'}")